# AbacusTabulation Minimal Usage

This notebook shows the minimal Python-side workflow after prepared catalogs, paircount tables, and the high-resolution HMF already exist.

Edit `CONFIG_PATH` if your run config lives somewhere else. The code can be run from either the repository root or the `notebooks/` directory.

In [ ]:
from pathlib import Path
import copy
import sys

import numpy as np

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "source"))

from abacustabulation import (
    HODClusteringTabulator,
    HODDerivedTabulator,
    hod_derived_quantities,
    hod_number_density,
    hod_satellite_fraction,
    load_config,
    projected_wp,
    resolve_paircount_path_from_config,
    smu_multipoles,
)

## Load Config and Choose HOD Parameters

The helper below starts from `hod.params`, applies tracer fixed parameters, and fills free fit parameters with their configured `initial` values.

In [ ]:
CONFIG_PATH = repo / "configs" / "lrg_fit_override.yaml"
TRACER = "LRG"

config = load_config(CONFIG_PATH)


def initial_hod_params(config, tracer):
    params = dict(config.get("hod", {}).get("params", {}))

    fit_config = config.get("fit", {})
    tracer_theory = fit_config.get("theory", {}).get("tracers", {}).get(tracer, {})
    params.update(tracer_theory.get("fixed_params", {}))

    for name, spec in fit_config.get("parameters", {}).items():
        if "." in name:
            tracer_name, param_name = name.split(".", 1)
            if tracer_name != tracer:
                continue
        else:
            param_name = name

        if isinstance(spec, dict):
            if "initial" in spec:
                params[param_name] = spec["initial"]
            else:
                params[param_name] = 0.5 * (spec["min"] + spec["max"])
        else:
            values = list(spec)
            params[param_name] = 0.5 * (values[0] + values[1])

    return params


hod_model = config.get("fit", {}).get("theory", {}).get("tracers", {}).get(TRACER, {}).get(
    "hod_model", config.get("hod", {}).get("model", "lrg")
)
hod_params = initial_hod_params(config, TRACER)

print("HOD model:", hod_model)
hod_params

## Paircount Tabulator: `wp(rp)` from `rppi`

`HODClusteringTabulator` loads one paircount table once and can then evaluate many HOD parameter sets cheaply.

In [ ]:
rppi_path = resolve_paircount_path_from_config(
    config,
    clustering="rppi",
    path_config={"job": "clustering"},
)
print(rppi_path)

rppi_tab = HODClusteringTabulator.from_paircount_file(rppi_path)
rppi_result = rppi_tab.correlation(hod_params, hod_model=hod_model)

rp_edges = rppi_result.paircounts.bins["rp_edges"]
rp_mid = np.sqrt(rp_edges[:-1] * rp_edges[1:])
wp = projected_wp(rppi_result)

print("number density from paircount table:", rppi_result.number_density)
print("rp bins:", rp_mid.shape[0])
print("wp first bins:", wp[:5])

## Paircount Tabulator: Multipoles from `smu`

In [ ]:
smu_path = resolve_paircount_path_from_config(
    config,
    clustering="smu",
    path_config={"job": "clustering"},
)
print(smu_path)

smu_tab = HODClusteringTabulator.from_paircount_file(smu_path)
smu_result = smu_tab.correlation(hod_params, hod_model=hod_model)

s_edges = smu_result.paircounts.bins["s_edges"]
s_mid = np.sqrt(s_edges[:-1] * s_edges[1:])
ells = smu_multipoles(smu_result, ells=(0, 2))

print("s bins:", s_mid.shape[0])
print("xi0 first bins:", ells[0][:5])
print("xi2 first bins:", ells[2][:5])

## HMF Object and HMF-Based HOD Quantities

For HMF quantities, use the high-resolution HMF rather than the coarser paircount mass bins.

In [ ]:
hmf_config = copy.deepcopy(config)
hmf_config.setdefault("derived", {})["quantities"] = [
    "n_cen",
    "n_sat",
    "number_density",
    "satellite_fraction",
    "log10_mh_cen_med",
    "log10_mh_sat_med",
]
hmf_config.setdefault("derived", {}).setdefault("linear_bias", {})["enabled"] = False

hmf_derived = HODDerivedTabulator.from_config(hmf_config, tracer=TRACER)
hmf = hmf_derived.hmf

print(type(hmf).__name__)
print("HMF bins:", hmf.logm_centers.size)
print("volume:", hmf.volume)
print("ng from HMF:", hod_number_density(hmf, hod_params, hod_model=hod_model))
print("fsat from HMF:", hod_satellite_fraction(hmf, hod_params, hod_model=hod_model))

## Derived Parameter Tabulator

`HODDerivedTabulator` is the reusable class for scalar derived quantities. Build it once, then call `evaluate` for any HOD parameter set.

In [ ]:
derived_values = hmf_derived.evaluate(hod_params)
derived_values

In [ ]:
# Convenience dataclass with the common HMF-only quantities.
hod_derived_quantities(hmf, hod_params, hod_model=hod_model)

## Optional: Linear Bias Derived Quantity

Set `USE_LINEAR_BIAS = True` when the real-space linear-bias `smu` paircount table exists and `cosmoprimo` is available in the environment.

In [ ]:
USE_LINEAR_BIAS = False

if USE_LINEAR_BIAS:
    bias_config = copy.deepcopy(config)
    bias_config.setdefault("derived", {})["quantities"] = ["linear_bias"]
    bias_config.setdefault("derived", {}).setdefault("linear_bias", {})["enabled"] = True

    bias_derived = HODDerivedTabulator.from_config(bias_config, tracer=TRACER)
    bias = bias_derived.linear_bias(hod_params, hod_model=hod_model)
    print("linear bias:", bias)
else:
    print("Set USE_LINEAR_BIAS = True to evaluate linear bias.")